In [7]:
import asyncio
import aiohttp
import re
from pathlib import Path

# ---------- CONFIG ----------
BASE_URL = "https://piece-climatisation.com"
START_URL = BASE_URL + "/boutique-piece-detachee-climatisation/fournisseur/ebara"

OUTPUT_DIR = Path("ebara_pages")
OUTPUT_DIR.mkdir(exist_ok=True)

CONCURRENT_REQUESTS = 10
RETRIES = 3
START_PAGE = 1   # 👈 resume from here

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
}


# ---------- HELPERS ----------
def safe_filename(page_num):
    return f"ebara_page_{page_num}.html"


def extract_last_page(html):
    matches = re.findall(r'page=(\d+)', html)
    return max(map(int, matches)) if matches else 1


# ---------- FETCH ----------
async def fetch(session, url):
    for attempt in range(RETRIES):
        try:
            async with session.get(url, headers=HEADERS, timeout=30) as res:
                if res.status == 200:
                    return await res.text()
                else:
                    print(f"Status {res.status}: {url}")
        except Exception as e:
            print(f"Retry {attempt+1} → {url} → {e}")

        await asyncio.sleep(1)

    return None


# ---------- SAVE ----------
def save_html(page_num, html):
    file_path = OUTPUT_DIR / safe_filename(page_num)

    # ⏭ Skip if already exists (resume-safe)
    if file_path.exists():
        print(f"⏭ Skipped (exists): page {page_num}")
        return

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Saved page {page_num}")


# ---------- WORKER ----------
async def scrape_page(session, page_num, semaphore):
    async with semaphore:
        file_path = OUTPUT_DIR / safe_filename(page_num)

        # ⏭ Skip before request (faster resume)
        if file_path.exists():
            print(f"⏭ Already downloaded: {page_num}")
            return

        url = f"{START_URL}?page={page_num}" if page_num > 1 else START_URL

        html = await fetch(session, url)
        if html:
            save_html(page_num, html)
        else:
            print(f"❌ Failed page {page_num}")


# ---------- MAIN ----------
async def main():
    connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS)
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

    async with aiohttp.ClientSession(connector=connector) as session:
        print("Fetching first page to detect total pages...")

        first_html = await fetch(session, START_URL)
        if not first_html:
            print("❌ Failed first page")
            return

        last_page = extract_last_page(first_html)
        print(f"Total pages: {last_page}")
        print(f"Resuming from page: {START_PAGE}")

        # Ensure start is valid
        start = max(1, START_PAGE)

        # Create tasks from resume point
        tasks = [
            scrape_page(session, i, semaphore)
            for i in range(start, last_page + 1)
        ]

        # Batch processing (important for 5000+ pages)
        batch_size = 150

        for i in range(0, len(tasks), batch_size):
            batch = tasks[i:i + batch_size]
            print(f"\n🚀 Batch {i + start} → {i + start + len(batch) - 1}")

            await asyncio.gather(*batch)

            # small delay to avoid blocking
            await asyncio.sleep(2)


if __name__ == "__main__":
    await main()

Fetching first page to detect total pages...
Total pages: 834
Resuming from page: 1

🚀 Batch 1 → 150
✅ Saved page 1
✅ Saved page 11
✅ Saved page 7
✅ Saved page 6
✅ Saved page 3
✅ Saved page 9
✅ Saved page 5
✅ Saved page 8
✅ Saved page 4
✅ Saved page 12
✅ Saved page 2
✅ Saved page 13
✅ Saved page 10
✅ Saved page 14
✅ Saved page 16
✅ Saved page 19
✅ Saved page 15
✅ Saved page 21
✅ Saved page 20
✅ Saved page 18
✅ Saved page 17
✅ Saved page 24
✅ Saved page 22
✅ Saved page 23
✅ Saved page 25
✅ Saved page 27
✅ Saved page 26
✅ Saved page 31
✅ Saved page 30
✅ Saved page 32
✅ Saved page 28
✅ Saved page 29
✅ Saved page 33
✅ Saved page 34
✅ Saved page 35
✅ Saved page 37
✅ Saved page 36
✅ Saved page 38
✅ Saved page 40
✅ Saved page 43
✅ Saved page 39
✅ Saved page 42
✅ Saved page 45
✅ Saved page 44
✅ Saved page 41
✅ Saved page 46
✅ Saved page 47
✅ Saved page 53
✅ Saved page 48
✅ Saved page 54
✅ Saved page 56
✅ Saved page 50
✅ Saved page 52
✅ Saved page 57
✅ Saved page 51
✅ Saved page 55
✅ Saved page

In [8]:
import shutil
from google.colab import files

zip_file_name = 'ebara_pages.zip'
shutil.make_archive(zip_file_name.replace('.zip', ''), 'zip', 'ebara_pages')

print(f'Successfully created {zip_file_name}. Downloading now...')
files.download(zip_file_name)

Successfully created ebara_pages.zip. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>